In [2]:
import SimpleITK as sitk
import numpy as np
import glob
import os
import re
import matplotlib.pyplot as plt
import sys
sys.path.append(os.path.abspath('..'))
import config

import numpy as np
import tifffile
import zarr
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, label
from skimage.morphology import white_tophat, disk

print("Environment initialized.")

TARGET_CORE = "Core_03"
DATA_BASE_PATH = os.path.join(config.DATASPACE, "Filter_AKAZE_RoMaV2_Linear_Warp_map/")
INPUT_FOLDER = os.path.join(DATA_BASE_PATH, TARGET_CORE)

if not os.path.exists(INPUT_FOLDER):
    raise FileNotFoundError(f"Input folder not found: {INPUT_FOLDER}")

print(f"Targeting Core: {TARGET_CORE}")
print(f"Data Path: {INPUT_FOLDER}")

SLICE_Z = 11
VOL_PATH = os.path.join(INPUT_FOLDER, f'{TARGET_CORE}_AKAZE_RoMaV2_Linear_Aligned.ome.tif')

print("Loading data via lazy extraction...")

with tifffile.TiffFile(VOL_PATH) as tif:
    lazy_vol = tif.aszarr()
    z_array = zarr.open(lazy_vol, mode='r')
    print(f"Volume shape mapped as: {z_array.shape}")
    
    N_CHANNELS = z_array.shape[1]
    raw_channels = [
        z_array[SLICE_Z, c, :, :].astype(np.float32)
        for c in range(N_CHANNELS)
    ]

print(f"Loaded {N_CHANNELS} channels. Extraction complete.")

Environment initialized.
Targeting Core: Core_03
Data Path: /data3/junming/3D-TMA-Register/Filter_AKAZE_RoMaV2_Linear_Warp_map/Core_03
Loading data via lazy extraction...
Volume shape mapped as: (19, 8, 6112, 6112)
Loaded 8 channels. Extraction complete.


In [ ]:
# =====================================================================
# 2. Dust-Aware Denoising Pipeline (All Channels) — OpenCV-accelerated
# =====================================================================
#
# Speed bottleneck in the original: skimage.white_tophat with a disk SE
# is O(r²) per pixel in pure Python/Cython on a 6112×6112 image.
#
# Two changes for ~10–20× speedup:
#
#   A) cv2.morphologyEx(MORPH_TOPHAT) — OpenCV's morphological ops are
#      SIMD-vectorised C and run the same algorithm in a fraction of the
#      time. We build an elliptical SE (equivalent to skimage disk).
#
#   B) cv2.medianBlur instead of scipy.ndimage.median_filter — also
#      OpenCV C, noticeably faster for ksize=3.
#
#   C) Parallel channels via concurrent.futures.ThreadPoolExecutor —
#      the OpenCV ops release the GIL, so threads give real parallelism
#      across channels with no extra memory copy overhead.
#
#   D) Vectorised dust-mask step: replace the Python loop over connected
#      components with cv2.connectedComponentsWithStats, which returns
#      a stats matrix we can threshold in one numpy operation.

import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

PIXEL_SCALE_UM = 0.496          # µm per pixel
NUCLEUS_RADIUS_UM = 5           # typical nucleus radius in µm
NUCLEUS_RADIUS_PX = int(round(NUCLEUS_RADIUS_UM / PIXEL_SCALE_UM))  # ~10 px

DUST_MIN_AREA_PX2 = np.pi * (NUCLEUS_RADIUS_PX * 2) ** 2
DUST_INTENSITY_PERCENTILE = 99

# Build the structuring element once — reused for every channel
SE_DIAM = 2 * NUCLEUS_RADIUS_PX + 1          # must be odd
_SE = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (SE_DIAM, SE_DIAM)    # equivalent to skimage disk(r)
)

print(f"Nucleus radius : {NUCLEUS_RADIUS_PX} px  ({NUCLEUS_RADIUS_UM} µm)")
print(f"SE diameter    : {SE_DIAM} px")
print(f"Dust min area  : {DUST_MIN_AREA_PX2:.0f} px²")


def denoise_channel(raw: np.ndarray,
                    se=_SE,
                    dust_min_area: float = DUST_MIN_AREA_PX2,
                    dust_intensity_pct: float = DUST_INTENSITY_PERCENTILE
                    ) -> dict:
    """
    OpenCV-accelerated dust-aware denoise for one 2-D float32 channel.
    """
    # cv2 needs uint16 for medianBlur on large images;
    # we normalise to [0, 65535], filter, then restore scale.
    raw_max = raw.max()
    if raw_max == 0:
        empty = np.zeros_like(raw)
        return dict(raw=raw, median=empty, tophat=empty,
                    dust_mask=empty.astype(bool), cleaned=empty,
                    log=empty, removed=empty)

    scale = 65535.0 / raw_max
    u16 = (raw * scale).clip(0, 65535).astype(np.uint16)

    # Step 1: Median filter (ksize must be odd; 3 = same as before)
    med_u16 = cv2.medianBlur(u16, 3)

    # Step 2: White Top-Hat  =  image - morphological_opening(image)
    tophat_u16 = cv2.morphologyEx(
        med_u16, cv2.MORPH_TOPHAT, se,
        borderType=cv2.BORDER_REFLECT
    )

    # Restore float32 scale
    med     = med_u16.astype(np.float32)    / scale
    tophat  = tophat_u16.astype(np.float32) / scale

    # Step 3: Vectorised dust-mask via cv2.connectedComponentsWithStats
    # Build a binary mask of the genuinely-bright top-hat pixels
    pos_pixels = tophat_u16[tophat_u16 > 0]
    if pos_pixels.size == 0:
        dust_mask = np.zeros(raw.shape, dtype=bool)
    else:
        bright_thresh_u16 = int(np.percentile(pos_pixels, dust_intensity_pct))
        bright_bin = (tophat_u16 >= bright_thresh_u16).astype(np.uint8)

        # connectedComponentsWithStats returns (n_labels, label_img, stats, centroids)
        # stats columns: LEFT, TOP, WIDTH, HEIGHT, AREA
        n_labels, label_img, stats, _ = cv2.connectedComponentsWithStats(
            bright_bin, connectivity=8
        )
        # Area is stats[:, cv2.CC_STAT_AREA]; index 0 = background
        large_ids = np.where(stats[1:, cv2.CC_STAT_AREA] >= dust_min_area)[0] + 1
        dust_mask = np.isin(label_img, large_ids)

    cleaned = tophat.copy()
    cleaned[dust_mask] = 0

    return {
        "raw":       raw,
        "median":    med,
        "tophat":    tophat,
        "dust_mask": dust_mask,
        "cleaned":   cleaned,
        "log":       np.log1p(cleaned),
        "removed":   raw - cleaned,
    }


# Parallel execution across channels
# max_workers=4 is a safe default; raise it if you have more CPU cores
# and enough RAM (each channel is ~150 MB at float32 for 6112²).
MAX_WORKERS = min(4, N_CHANNELS)

print(f"\nRunning denoising on {N_CHANNELS} channels "
      f"with {MAX_WORKERS} parallel workers...")
t0 = time.perf_counter()

results = [None] * N_CHANNELS
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(denoise_channel, ch): i
               for i, ch in enumerate(raw_channels)}
    for fut in as_completed(futures):
        idx = futures[fut]
        results[idx] = fut.result()
        print(f"  Ch {idx} done")

elapsed = time.perf_counter() - t0
print(f"\nAll channels processed in {elapsed:.1f}s "
      f"({elapsed/N_CHANNELS:.2f}s per channel)")

In [ ]:
# =====================================================================
# 2a. DIAGNOSTIC — Raw Intensity Histograms (run this first)
# =====================================================================
# Before estimating object sizes we need to understand each channel's
# intensity distribution. This tells us:
#   • Whether the signal is sparse (most pixels = 0/background) or dense
#   • Where a sensible foreground threshold sits
#   • Whether Otsu will behave (needs a clear bimodal distribution)
#
# We plot THREE things per channel on the same axes:
#   1. Full log1p intensity histogram (gray) — shows overall distribution
#   2. Otsu threshold (red dashed line) — where skimage would cut
#   3. A set of fixed percentile thresholds (50th, 90th, 99th) as
#      vertical lines — lets you see what fraction is truly signal
#
# What to look for:
#   - GOOD for CC method: clear bimodal histogram, Otsu lands in the
#     valley, ~5–40% pixels above threshold
#   - BAD / needs FFT: histogram is unimodal or Otsu is way up in the
#     tail (FG fraction <1% or >60%)
#   - Nearly empty channel: almost all zeros, histogram is a spike at 0

from skimage.filters import threshold_otsu

n_cols = 4
n_rows = int(np.ceil(N_CHANNELS / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten()

for i, raw in enumerate(raw_channels):
    ax = axes[i]
    flat = raw.ravel()

    # Subsample for speed on large images
    if flat.size > 2_000_000:
        flat = flat[::flat.size // 2_000_000]

    nonzero = flat[flat > 0]
    fg_frac_zero = (flat > 0).mean()   # fraction of non-zero pixels

    # log1p histogram of ALL pixels including zeros
    ax.hist(np.log1p(flat), bins=200, color='lightgray',
            edgecolor='none', label='All pixels')

    # Otsu on the raw values
    try:
        otsu_t = threshold_otsu(flat)
        fg_frac_otsu = (flat >= otsu_t).mean()
        ax.axvline(np.log1p(otsu_t), color='crimson', lw=1.5, ls='--',
                   label=f'Otsu={otsu_t:.0f} (FG={fg_frac_otsu:.1%})')
    except Exception:
        fg_frac_otsu = np.nan

    # Percentile reference lines
    for pct, col in [(50, 'steelblue'), (90, 'orange'), (99, 'purple')]:
        v = np.percentile(flat, pct)
        ax.axvline(np.log1p(v), color=col, lw=1, ls=':',
                   label=f'p{pct}={v:.0f}')

    ax.set_title(f'Ch {i}  |  non-zero={fg_frac_zero:.1%}')
    ax.set_xlabel('log1p(intensity)')
    ax.set_ylabel('Pixel count')
    ax.set_yscale('log')
    ax.legend(fontsize=6.5)

for i in range(N_CHANNELS, len(axes)):
    axes[i].set_visible(False)

fig.suptitle(
    'Raw Intensity Distributions per Channel\n'
    '(Use this to judge whether Otsu / CC or FFT is appropriate)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("Per-channel summary:")
print(f"{'Ch':>3}  {'non-zero px':>12}  {'Otsu thresh':>12}  {'Otsu FG%':>10}  {'p50':>8}  {'p90':>8}  {'p99':>8}")
for i, raw in enumerate(raw_channels):
    flat = raw.ravel()
    nz   = (flat > 0).mean()
    p50, p90, p99 = np.percentile(flat, [50, 90, 99])
    try:
        ot = threshold_otsu(flat)
        fg = (flat >= ot).mean()
        ot_str = f'{ot:>12.1f}'
        fg_str = f'{fg:>10.1%}'
    except Exception:
        ot_str, fg_str = '           —', '         —'
    print(f"{i:>3}  {nz:>12.1%}  {ot_str}  {fg_str}  {p50:>8.1f}  {p90:>8.1f}  {p99:>8.1f}")

In [ ]:
# =====================================================================
# 3. Per-Channel Visualization
# =====================================================================
# For each channel: Raw | Top-Hat | Dust mask | Cleaned | Removed

for ch_idx, res in enumerate(results):
    fig, axes = plt.subplots(1, 5, figsize=(30, 6))
    fig.suptitle(f"Channel {ch_idx} — Dust-Aware Denoising", fontsize=14, fontweight='bold')

    raw = res["raw"]
    tophat = res["tophat"]
    cleaned = res["cleaned"]
    removed = res["removed"]
    dust_mask = res["dust_mask"]

    vmax_raw = np.percentile(raw, 99.9)
    vmax_clean = np.percentile(cleaned[cleaned > 0], 99.9) if cleaned.max() > 0 else 1

    axes[0].imshow(raw, cmap='gray', vmax=vmax_raw)
    axes[0].set_title('1. Raw')
    axes[0].axis('off')

    axes[1].imshow(tophat, cmap='gray', vmax=vmax_clean)
    axes[1].set_title(f'2. Top-Hat (r={NUCLEUS_RADIUS_PX}px)')
    axes[1].axis('off')

    axes[2].imshow(dust_mask, cmap='Reds')
    axes[2].set_title('3. Detected Dust Mask')
    axes[2].axis('off')

    axes[3].imshow(cleaned, cmap='gray', vmax=vmax_clean)
    axes[3].set_title('4. Cleaned')
    axes[3].axis('off')

    axes[4].imshow(removed, cmap='inferno', vmax=np.percentile(removed, 99.9))
    axes[4].set_title('5. Removed (artifacts + background)')
    axes[4].axis('off')

    plt.tight_layout()
    plt.show()
    print(f"  Ch {ch_idx}: dust regions found = {dust_mask.sum()} px, "
          f"signal retained = {(cleaned > 0).sum()} px")

In [ ]:
# =====================================================================
# 4. Intensity Distribution — Before vs After (All Channels)
# =====================================================================

n_cols = 4
n_rows = int(np.ceil(N_CHANNELS / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten()

for ch_idx, res in enumerate(results):
    ax = axes[ch_idx]
    raw_pos = res["raw"][res["raw"] > 0].ravel()
    clean_pos = res["cleaned"][res["cleaned"] > 0].ravel()
    ax.hist(np.log1p(raw_pos), bins=100, alpha=0.5, color='tomato', label='Raw')
    ax.hist(np.log1p(clean_pos), bins=100, alpha=0.5, color='steelblue', label='Cleaned')
    ax.set_title(f'Channel {ch_idx}')
    ax.set_xlabel('log1p(intensity)')
    ax.set_yscale('log')
    ax.legend(fontsize=8)

# Hide any unused subplots
for i in range(N_CHANNELS, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Intensity Distribution: Raw vs Cleaned (all channels)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()